[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alinespereira/SME0211/blob/main/notebooks/01_simplex.ipynb)

# O método simplex, passo a passo

Exemplos de uso do pacote `simplex`: o problema como escrito no papel, a forma padrão, a Fase 1, o tableau e o `Solver`, uma iteração de cada vez. A teoria está na documentação (`make docs`).

A célula abaixo instala os pacotes `simplex` e `printing_company` direto do GitHub (no Colab, ou em qualquer ambiente sem o projeto). Rodando localmente, com `uv sync`, ela pode ser pulada.

In [ ]:
!pip install -q git+https://github.com/alinespereira/SME0211

## 1. O problema no papel

$$
\begin{aligned}
\max \quad & 3 x_1 + 2 x_2 \\
\text{s.a.} \quad & x_1 + x_2 \leq 4, \\
& x_1 + 3 x_2 \leq 6, \\
& x_1 \geq 1, \\
& x \geq 0.
\end{aligned}
$$

In [2]:
from simplex import Constraint, PivotingRule, Problem, Solver, Tableau

p = Problem(
    sense="max",
    c=[3, 2],
    constraints=(
        Constraint(coefs=[1, 1], sense="<=", rhs=4),
        Constraint(coefs=[1, 3], sense="<=", rhs=6),
        Constraint(coefs=[1, 0], sense=">=", rhs=1),
    ),
)
print(p)

max   3 x1 + 2 x2
s.a.    x1 +   x2 <= 4
        x1 + 3 x2 <= 6
        x1        >= 1
      x1, x2 >= 0


In [3]:
solution = p.solve()
solution.status, solution.x, solution.objective

(<Status.OPTIMAL: 'optimal'>, array([4., 0.]), 12.0)

O mesmo problema pode ser lido de um arquivo TOML, com as mesmas chaves que
os campos de `Problem` (veja `examples/pequeno.toml` no repositório):

In [4]:
from pathlib import Path

Path("pequeno.toml").write_text(
    """
sense = "max"
c = [3, 2]
names = ["x1", "x2"]

[[constraints]]
coefs = [1, 1]
sense = "<="
rhs = 4

[[constraints]]
coefs = [1, 3]
sense = "<="
rhs = 6

[[constraints]]
coefs = [1, 0]
sense = ">="
rhs = 1
"""
)
Problem.from_toml(path="pequeno.toml").solve().objective

12.0

## 2. Forma padrão

O $\max$ vira $\min$ (custos com sinal trocado), as restrições $\leq$
ganham as folgas `s1` e `s2` e a restrição $\geq$ ganha o excesso `e3`.

In [5]:
sf = p.to_standard_form()
print(sf.names)
print(sf.A)
print(sf.c, sf.objective_sign)

('x1', 'x2', 's1', 's2', 'e3')
[[ 1.  1.  1.  0.  0.]
 [ 1.  3.  0.  1.  0.]
 [ 1.  0.  0.  0. -1.]]
[-3. -2.  0.  0.  0.] -1


## 3. Fase 1

O problema auxiliar começa com as artificiais `y1`, `y2` e `y3` (colunas 5,
6 e 7) na base e termina numa base viável só com colunas da forma padrão.

In [6]:
phase_one = sf.phase_one_tableau()
print(phase_one.basis, phase_one.variable_names[5:])

t0 = sf.initial_tableau(solver=Solver())
print(t0.basis, sf.original_solution(x=t0.solution))

(5, 6, 7) ('y1', 'y2', 'y3')
(2, 1, 0) [1.         1.66666667]


## 4. O tableau

$$
\begin{aligned}
\min \quad & -x_1 - 2 x_2 \\
\text{s.a.} \quad & x_1 + x_2 + s_1 = 4, \\
& x_1 + 3 x_2 + s_2 = 6, \\
& x \geq 0,
\end{aligned}
$$

na base das folgas $s_1$ e $s_2$ (índices 2 e 3 no código).

In [7]:
t = Tableau(
    A=[[1, 1, 1, 0], [1, 3, 0, 1]],
    b=[4, 6],
    c=[-1, -2, 0, 0],
    basis=(2, 3),
    names=("x1", "x2", "s1", "s2"),
)
print("x_B =", t.basic_solution)
print("x =", t.solution)
print("custos reduzidos =", t.reduced_costs)
print("coluna de x2 =", t.column(j=1))
print(t.table)

x_B = [4. 6.]
x = [0. 0. 4. 6.]
custos reduzidos = [-1. -2.  0.  0.]
coluna de x2 = [1. 3.]
[[ 0. -1. -2.  0.  0.]
 [ 4.  1.  1.  1.  0.]
 [ 6.  1.  3.  0.  1.]]


`render_tableau` mostra o mesmo tableau com nomes, destacando a variável que
entra (↓), a que sai (→) e o pivô (entre colchetes):

In [8]:
from simplex.display import render_tableau

print(render_tableau(tableau=t, entering=1, leaving=1).plain)

        │ x1  x2 ↓ s1  s2
────────┼─────────────────
  -z 0  │ -1   -2   0   0
────────┼─────────────────
  s1 4  │  1    1   1   0
→ s2 6  │  1   [3]  0   1


Na base $\lbrace x_1, x_2 \rbrace$, todos os custos reduzidos são não
negativos: é a base ótima. O tableau original não muda.

In [9]:
otimo = t.with_basis(basis=(0, 1))
print(render_tableau(tableau=otimo).plain)
print("x =", otimo.solution, "custo =", otimo.objective_value)
print("base original:", t.basis)

        │ x1  x2    s1    s2
────────┼────────────────────
  -z 5  │  0   0   0.5   0.5
────────┼────────────────────
  x1 3  │  1   0   1.5  -0.5
  x2 1  │  0   1  -0.5   0.5
x = [3. 1. 0. 0.] custo = -5.0
base original: (2, 3)


## 5. O `Solver`, um passo de cada vez

Com a regra de Bland, entra o menor $j$ com custo reduzido negativo, e sai,
entre as linhas que empatam no teste da razão, a de menor $B(\ell)$.

In [10]:
solver = Solver(rule=PivotingRule.BLAND)

print("ótimo?", solver.is_optimal(tableau=t))
j = solver.entering_variable(tableau=t)
u = t.column(j=j)
ell = solver.leaving_row(tableau=t, u=u)  # razões 4/1 e 6/1
print(f"entra {t.variable_names[j]}, sai {t.variable_names[t.basis[ell]]}")
print(render_tableau(tableau=t, entering=j, leaving=ell).plain)
print("nova base:", solver.change_basis(tableau=t, j=j, ell=ell).basis)

ótimo? False
entra x1, sai s1
        │ x1 ↓ x2  s1  s2
────────┼─────────────────
  -z 0  │  -1  -2   0   0
────────┼─────────────────
→ s1 4  │  [1]  1   1   0
  s2 6  │   1   3   0   1
nova base: (0, 3)


O método completo:

In [11]:
result = solver.solve(tableau=t)
(
    result.status,
    result.iterations,
    result.tableau.solution,
    result.tableau.objective_value,
)

(<Status.OPTIMAL: 'optimal'>, 2, array([3., 1., 0., 0.]), -5.0)

Com o `logging` no nível DEBUG, o `Solver` mostra o tableau de cada
iteração, com a variável que entra, a que sai e o pivô em destaque:

In [12]:
from simplex import configure_logging

configure_logging(debug=True)
solver.solve(tableau=t)
configure_logging(debug=False)

[22:13:13] DEBUG    Iteração 0: entra x1 (↓), sai s1 (→), pivô [1]                                     ]8;id=10917716;file:///home/user/repo/src/simplex/solver.py\solver.py]8;;\:]8;id=10917717;file:///home/user/repo/src/simplex/solver.py#40\40]8;;\

        │ x1 ↓ x2  s1  s2
────────┼─────────────────
  -z 0  │  -1  -2   0   0
────────┼─────────────────
→ s1 4  │  [1]  1   1   0
  s2 6  │   1   3   0   1

           DEBUG    Iteração 1: entra x2 (↓), sai s2 (→), pivô [2]                                     ]8;id=10917722;file:///home/user/repo/src/simplex/solver.py\solver.py]8;;\:]8;id=10917723;file:///home/user/repo/src/simplex/solver.py#40\40]8;;\

        │ x1  x2 ↓ s1  s2
────────┼─────────────────
  -z 4  │  0   -1   1   0
────────┼─────────────────
  x1 4  │  1    1   1   0
→ s2 2  │  0   [2] -1   1

           DEBUG    Iteração 2: solução ótima                                                          ]8;id=10917728;file:///home/user/repo/src/simplex/solver.py\solver.py]8;;\:]8;id=10917729;file:///home/user/repo/src/simplex/solver.py#40\40]8;;\

        │ x1  x2    s1    s2
────────┼────────────────────
  -z 5  │  0   0   0.5   0.5
────────┼────────────────────
  x1 3  │  1   0   1.5  -0.5
  x2 1  │  0   1  -0.5   0.5

## 6. Ciclagem: o exemplo de Beale

Neste problema degenerado (Bertsimas e Tsitsiklis, exemplo 3.6), o simplex
com a regra do maior custo reduzido cicla. Com Bland ou com a regra
lexicográfica, ele termina no ótimo, com custo $-5/4$.

In [13]:
beale = Tableau(
    A=[
        [1, 0, 0, 1 / 4, -8, -1, 9],
        [0, 1, 0, 1 / 2, -12, -1 / 2, 3],
        [0, 0, 1, 0, 0, 1, 0],
    ],
    b=[0, 0, 1],
    c=[0, 0, 0, -3 / 4, 20, -1 / 2, 6],
    basis=(0, 1, 2),
)
for rule in PivotingRule:
    r = Solver(rule=rule).solve(tableau=beale)
    print(
        f"{rule:>13}: {r.status}, {r.iterations} iterações, custo {r.tableau.objective_value:.4g}"
    )

        bland: optimal, 6 iterações, custo -1.25
lexicographic: optimal, 2 iterações, custo -1.25


## 7. Um problema inviável

Com $x_1 \leq 1$ e $x_1 \geq 2$, a Fase 1 termina com custo positivo, e
não há solução.

In [14]:
inviavel = Problem(
    c=[1],
    constraints=(
        Constraint(coefs=[1], sense="<=", rhs=1),
        Constraint(coefs=[1], sense=">=", rhs=2),
    ),
)
s = inviavel.solve()
s.status, s.x, s.objective

(<Status.INFEASIBLE: 'infeasible'>, None, None)